# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Pretty-print the metadata description
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The dataset's Croissant schema contains three main tables reflecting clinical, laboratory, and genetic data. To enumerate the record sets and their fields, we use the metadata: each entity is referenced by its `@id`.

In [ ]:
# List all record sets and their fields referenced by @id
record_sets = dataset.record_sets

print("Available record sets (by @id):")
for rs in record_sets:
    print(f"---- RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '[Unnamed]')}")
    print("  Fields (@id):")
    for f in rs.fields:
        print(f"    - {f.id} ({getattr(f, 'name', '[Unnamed]')})")
    print("  Columns (@id):")
    for col in rs.columns:
        print(f"    - {col.id} ({getattr(col, 'name', '[Unnamed]')})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We reference record sets and fields by their `@id` as listed above.

Below, you will see how to extract all record sets, using their `@id`, into pandas DataFrames for convenient analysis.

In [ ]:
# Extract data from each record set by @id
# First, gather the @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show available DataFrames and their columns
for rs_id, df in dataframes.items():
    print(f"RecordSet @id: {rs_id}")
    print("Columns (by @id):", df.columns.tolist())
    print("---- First few rows:")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Let's analyze one of the record sets. We'll select a clinical numeric field for further exploration, filter records, normalize the values, and group by a categorical attribute. All fields are referenced by their `@id`.

Choose the record set and fields from the previous list:

In [ ]:
# Example: Select a record set containing clinical features
# Replace these example @id values with your actual @id from the overview
# For illustration, suppose record_set @id is 'cr:RecordSet_clinical', numeric_field @id is 'cr:field_age', group_field @id is 'cr:field_infertility'

# Try to locate appropriate ids:
clinical_rs = None
numeric_field = None
group_field = None
for rs in dataset.record_sets:
    if 'clinical' in getattr(rs, 'name', '').lower():
        clinical_rs = rs.id
        for f in rs.fields:
            if 'age' in getattr(f, 'name', '').lower():
                numeric_field = f.id
            if 'infertility' in getattr(f, 'name', '').lower():
                group_field = f.id
        break

# If not found, fallback to first record set and numeric column
if clinical_rs is None:
    clinical_rs = record_set_ids[0]
    # Try to pick first numeric column
    df0 = dataframes[clinical_rs]
    for col in df0.columns:
        # Try to guess if numeric
        try:
            if pd.api.types.is_numeric_dtype(df0[col]):
                numeric_field = col
                break
        except Exception:
            pass
    # Try to guess a group field (categorical)
    for col in df0.columns:
        if pd.api.types.is_string_dtype(df0[col]):
            group_field = col
            break

# Proceed with the chosen record set and fields
print(f"Selected record set @id: {clinical_rs}")
print(f"Numeric field @id: {numeric_field}")
print(f"Group field @id: {group_field}")

df = dataframes[clinical_rs]

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by key attribute
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(6, 4))
sns.histplot(df[numeric_field], bins=10, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If group field is available, show a boxplot
if group_field in df.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f'{numeric_field} by {group_field} group')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
We successfully loaded the FAIR^2 dataset with `mlcroissant`, explored its record structure and fields referenced by their `@id`, and performed basic filtering, normalization, grouping, and visualization. As demonstrated, referencing entities using `@id` ensures reproducibility and clarity when working with Croissant-based datasets.

The FAIR^2 dataset's limited cohort illustrates genotype–phenotype heterogeneity and its relevance for clinical research, but is not suitable for broad predictive modeling. Further exploration can extend to laboratory and genetic data tables.